In [ ]:
import os
from pathlib import Path 

import tqdm
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from benchmark_data_leakage.chembl_requester import ChEMBLRequester
from benchmark_data_leakage.utils import find_repo_root, invert_inequality, calc_neglog10_val

repo_root = find_repo_root()
data_dir = repo_root / "data"
out_dir = data_dir / "out"
fig_dir = repo_root / "figures"
fig_dir.mkdir(exist_ok=True)

### Load the data we will use in the analyses

In [ ]:
# Get all ChEMBL activity data
with open(repo_root / "chembl_db.yaml") as f:
    chembl_db = yaml.safe_load(f)
chembl_req = ChEMBLRequester(**chembl_db)

chembl_protein_targets = chembl_req.get_single_protein_targets()
chembl_protein_targets = pd.DataFrame(chembl_protein_targets)

# Querying all activity data. This consumes ~12 GB of RAM. constrain to only your 
activity_data = chembl_req.get_all_single_protein_activity_data(
    # Constraining by target_chembl_id's is optional.
    # None = No constraint, this uses 12GB of data and takes a few minutes
    # ['CHEMBL123', 'CHEMBL456'] = only query those targets
    target_chembl_ids=None,
)
activity_data = pd.DataFrame(activity_data)

# Join some extra columns we want later
data_split = pd.read_csv(out_dir / "FEP_data_split.csv").set_index("uniprot_id")["data_split"]
chembl_protein_targets = chembl_protein_targets.join(
    data_split,
    on="uniprot_id"
)
activity_data = activity_data.join(
    chembl_protein_targets.set_index("chembl_target_id")[["gene_name", "organism", "data_split"]],
    on="chembl_target_id"
)

# Load the FEPp benchmark data
benchmark_data = pd.read_csv(out_dir / "FEPp_benchmark.csv")

### Analyse how often ligands in the benchmark appears in test and train/val

In [ ]:
gene_name = "MAPK8"   # CDK2, TYK2, MAPK8, MAPK14
benchmark_slice = benchmark_data.loc[
    benchmark_data["gene_name"] == gene_name
]

n_ligands = len(benchmark_slice)
n_ligands_with_chembl_ids = (~benchmark_slice["chembl_ligand_id"].isna()).sum()
ligand_ids = benchmark_slice["chembl_ligand_id"].values

In [ ]:
print(f"{gene_name} appears with {n_ligands} compounds in the FEP+ 4 test set")
print(f"Of those {n_ligands_with_chembl_ids} compounds exist in ChEMBL")
print(f"In ChEMBL they appear against different targets this many times:")
activity_data.loc[
    activity_data["chembl_ligand_id"].isin(ligand_ids)
].drop_duplicates(
    ["gene_name", "chembl_ligand_id"]
).value_counts(["data_split", "gene_name", "organism"]).iloc[:20]

### Create figures and tables for slides (JNK1 / MAPK8)

In [ ]:
df_bench_slice = benchmark_data.loc[benchmark_data["gene_name"] == "MAPK8"].copy()
def get_pIC50(row: pd.Series) -> float:
    if row["measurement_unit"] == "uM":
        return -np.log10(row["measurement_value"]) + 6
    elif row["measurement_unit"] == "nM":
        return -np.log10(row["measurement_value"]) + 9
df_bench_slice["pIC50"] = df_bench_slice.apply(get_pIC50, axis=1)
df_bench_slice = df_bench_slice[["ligand_name", "pIC50", "chembl_ligand_id", "chembl_target_id"]].rename(columns={
    "pIC50": "FEP+_pIC50",
    "ligand_name": "FEP+_ligand_name"
})

focus_assays = ["CHEMBL860181", "CHEMBL860184"]
activity_data_slice = activity_data.loc[
    activity_data["chembl_assay_id"].isin(focus_assays)
    & (activity_data["measurement_type"] == "IC50")
    & (activity_data["chembl_ligand_id"].isin(df_bench_slice["chembl_ligand_id"].values))
].copy()
# Fill missing pchembl_value
activity_data_slice["pchembl_value"] = activity_data_slice["pchembl_value"].fillna(
    activity_data_slice.apply(calc_neglog10_val, axis=1)
).astype(float)
# Creating relation for pIC50 values (e.g. invert the normal relation)
activity_data_slice["pval_relation"] = activity_data_slice["measurement_relation"].apply(invert_inequality)

df_combined = activity_data_slice.join(
    df_bench_slice.set_index("chembl_ligand_id")["FEP+_ligand_name"],
    on="chembl_ligand_id"
).join(
    df_bench_slice.set_index(["chembl_ligand_id", "chembl_target_id"])["FEP+_pIC50"],
    on=["chembl_ligand_id", "chembl_target_id"]
)
def include_rel_in_val(row: pd.Series) -> str:
    if row["pval_relation"] == "=":
        return f'{row["pchembl_value"]:.2f}'
    elif row["pval_relation"] == "<":
        return f'<{row["pchembl_value"]:.2f}'
    raise RuntimeError

df_combined["pIC50 (str)"] = df_combined.apply(include_rel_in_val, axis=1)

In [ ]:
# 1. Set the visual style and scaling for a slide/paper
import scipy.stats
import numpy as np
import matplotlib.pyplot as plt

df = df_combined.pivot(
    columns=["gene_name", "chembl_assay_id"],
    values="pchembl_value",
    index=["FEP+_ligand_name", "chembl_ligand_id"]
)
x_col = ("MAPK8", "CHEMBL860181")
y_col = ("MAPK9", "CHEMBL860184")
x_label = "Test set: JNK1 [pIC50]"
y_label = "Train set: JNK2 [pIC50]"

x = df[x_col].values
y = df[y_col].values
corr, _ = scipy.stats.pearsonr(x, y)

slope, intercept, *_ = scipy.stats.linregress(x, y)
x_fit = np.linspace(x.min() - 0.15, x.max() + 0.15, 200)
y_fit = slope * x_fit + intercept

# 95% CI band
n = len(x)
x_mean = x.mean()
se_fit = np.sqrt(np.sum((y - (slope * x + intercept))**2) / (n - 2)) * \
         np.sqrt(1/n + (x_fit - x_mean)**2 / np.sum((x - x_mean)**2))
t_crit = scipy.stats.t.ppf(0.975, n - 2)

fig, ax = plt.subplots(figsize=(4.5, 4.5))

ax.fill_between(x_fit, y_fit - t_crit * se_fit, y_fit + t_crit * se_fit,
                color='#3a6ea5', alpha=0.12, zorder=1)
ax.plot(x_fit, y_fit, color='#3a6ea5', linewidth=1.5, alpha=0.75, zorder=2)
ax.scatter(x, y, s=55, color='#1a3a5c', alpha=0.9, zorder=3, linewidths=0)

ax.text(0.06, 0.91, f'$r = {corr:.2f}$', transform=ax.transAxes,
        fontsize=12,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=2))

ax.set_xlabel('Test set: JNK1 [pIC$_{50}$]', fontsize=11)
ax.set_ylabel('Train set: JNK2 [pIC$_{50}$]', fontsize=11)
ax.tick_params(labelsize=9)
ax.grid(visible=True, which='major', linestyle=':', linewidth=0.5, color='#cccccc')
ax.set_axisbelow(True)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(fig_dir / "jnk1_jnk2_leakage_plot.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create data for table
df_for_table = df_combined.pivot(
    columns=["gene_name", "chembl_assay_id"],
    values="pIC50 (str)",
    index=["FEP+_ligand_name", "chembl_ligand_id"],
    ).sort_values(("MAPK8", "CHEMBL860181"))
df_for_table

In [ ]:
# Convert to latex rows

def pic50_to_rgb(value: float, vmin: float = 5.0, vmax: float = 8.0) -> str:
    norm = np.clip((value - vmin) / (vmax - vmin), 0, 1)
    cmap = plt.get_cmap('RdYlGn')
    r, g, b, _ = cmap(norm)
    return f"{r:.2f},{g:.2f},{b:.2f}"


def parse_pic50(value: str) -> float:
    """Handle <5.0 and normal float strings."""
    if isinstance(value, str) and value.startswith('<'):
        return float(value[1:])
    return float(value)


def df_to_latex_rows(df) -> str:
    rows = []
    for (lig_name, chembl_id), row in df.iterrows():
        cells = [lig_name.replace("_", "\\_"), chembl_id]
        for col in df.columns:
            raw = row[col]
            numeric = parse_pic50(raw)
            color = pic50_to_rgb(numeric)
            cells.append(f"\\cellcolor[rgb]{{{color}}} {raw}")
        rows.append(" & ".join(cells) + " \\\\")
    return "\n".join(rows)

print(df_to_latex_rows(df_for_table))

### Calculate sequence identify of JNK1 / JNK2

In [ ]:
# Look up UniProt IDs for MAPK8 (JNK1) and MAPK9 (JNK2)
uniprot_jnk1 = chembl_protein_targets.loc[
    chembl_protein_targets["gene_name"] == "MAPK8", "uniprot_id"
].iloc[0]
uniprot_jnk2 = chembl_protein_targets.loc[
    chembl_protein_targets["gene_name"] == "MAPK9", "uniprot_id"
].iloc[0]
print(f"JNK1 (MAPK8) UniProt: {uniprot_jnk1}")
print(f"JNK2 (MAPK9) UniProt: {uniprot_jnk2}")

# Load sequences from the FASTA used for MMseqs2 clustering
from Bio import SeqIO
from Bio.Align import PairwiseAligner

# NOTE: you need to run notebooks/01_create_data_split.ipynb to have the following file
fasta_path = data_dir / "processed" / "fasta_seqs_for_clustering.fasta"
seqs = SeqIO.to_dict(SeqIO.parse(fasta_path, "fasta"))

jnk1_seq = str(seqs[uniprot_jnk1].seq)
jnk2_seq = str(seqs[uniprot_jnk2].seq)

# Global pairwise alignment (Needleman-Wunsch)
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 1
aligner.mismatch_score = 0
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

best_alignment = next(iter(aligner.align(jnk1_seq, jnk2_seq)))
aligned_jnk1, aligned_jnk2 = best_alignment[0], best_alignment[1]

identical = sum(a == b for a, b in zip(aligned_jnk1, aligned_jnk2))
align_len = len(aligned_jnk1)
pct_identity = 100 * identical / align_len

print(f"\nJNK1 length: {len(jnk1_seq)} aa")
print(f"JNK2 length: {len(jnk2_seq)} aa")
print(f"Alignment length: {align_len} aa")
print(f"Sequence identity: {pct_identity:.1f}%")

### Figures for OpenFE targets

In [ ]:
benchmark_data = pd.read_csv(out_dir / "OpenFE_benchmark.csv")

In [ ]:
import scipy.stats
import numpy as np
import matplotlib.pyplot as plt

pairs = [
    # assay_id, test_assay_id, train_assay_id, train_measurement_type, sequence_identity
    # Note: use above to calculate sequence identity
    ("merck-tnks2", "TNKS2", "CHEMBL4322251", "IC50", 71.4),
    ("miscellaneous_set-btk", "BTK", "CHEMBL3588238", "IC50", 22.5),
]

i = 0

test_assay_id = pairs[i][0]
test_gene = pairs[i][1]
train_assay_id = pairs[i][2]
train_measurement_type = pairs[i][3]
sequence_identity = pairs[i][4]

df_bench_slice = benchmark_data.loc[
    benchmark_data["assay_id"] == test_assay_id
].copy()

activity_data_slice = activity_data.loc[
    (activity_data["chembl_assay_id"] == train_assay_id)
    & (activity_data["measurement_type"] == train_measurement_type)
    & (activity_data["measurement_relation"] == "=")
    & (activity_data["chembl_ligand_id"].isin(df_bench_slice["chembl_ligand_id"].values))
].copy()

df_joined = df_bench_slice.join(
    activity_data_slice.set_index("chembl_ligand_id")["pchembl_value"],
    on="chembl_ligand_id",
    rsuffix="_train"
)

slice_in_train = ~df_joined["pchembl_value_train"].isna()
n_in_train = slice_in_train.sum()
n_total = len(slice_in_train)

system_group = test_assay_id.split("-")[0]
x_label = f"Test set: {system_group} – {test_gene} ($-\Delta G$)"
train_gene = activity_data_slice.iloc[0]["gene_name"]
y_label = f"Train set: {train_assay_id} – {train_gene} (pIC$_{{50}}$)"

x = -df_joined.loc[slice_in_train, "exp_dg_kcal_mol"].values
y = df_joined.loc[slice_in_train, "pchembl_value_train"].astype(float).values
corr, _ = scipy.stats.pearsonr(x, y)

slope, intercept, *_ = scipy.stats.linregress(x, y)
x_fit = np.linspace(x.min() - 0.15, x.max() + 0.15, 200)
y_fit = slope * x_fit + intercept

# 95% CI band
n = len(x)
x_mean = x.mean()
se_fit = np.sqrt(np.sum((y - (slope * x + intercept))**2) / (n - 2)) * \
         np.sqrt(1/n + (x_fit - x_mean)**2 / np.sum((x - x_mean)**2))
t_crit = scipy.stats.t.ppf(0.975, n - 2)

fig, ax = plt.subplots(figsize=(4.5, 4.5))

ax.fill_between(x_fit, y_fit - t_crit * se_fit, y_fit + t_crit * se_fit,
                color='#3a6ea5', alpha=0.12, zorder=1)
ax.plot(x_fit, y_fit, color='#3a6ea5', linewidth=1.5, alpha=0.75, zorder=2)
ax.scatter(x, y, s=55, color='#1a3a5c', alpha=0.9, zorder=3, linewidths=0)

ax.text(0.06, 0.91, f'$r = {corr:.2f}$\n$s = {sequence_identity:.1f}\%$\n$c = {n_in_train} / {n_total}$', transform=ax.transAxes,
        fontsize=12,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=2))
ax.set_xlabel(x_label, fontsize=11)
ax.set_ylabel(y_label, fontsize=11)
ax.tick_params(labelsize=9)
ax.grid(visible=True, which='major', linestyle=':', linewidth=0.5, color='#cccccc')
ax.set_axisbelow(True)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(fig_dir / f"OpenFE_{test_gene}_leakage_plot.png", dpi=300, bbox_inches='tight')
plt.show()